# Lab 06 — Manual Retrieval: What Breaks and Why

**Knowledge base:** `knowledge_base/03_retrieval/01_retriever_architecture.md`

**Concepts:** keyword overlap · vocabulary mismatch · retrieval failures · motivation for Module 02

This lab exposes the limits of naive keyword retrieval so you understand exactly what
BM25 and semantic search (Module 02) were built to solve.

---

In [ ]:
import os, sys
sys.path.insert(0, ".")
from dotenv import load_dotenv
from utils import generate_with_single_input

load_dotenv(override=True)
print(f"✅ Backend: {os.environ.get('LLM_BACKEND', 'ollama')}")

---
## 1 — The naive retriever from lab05

A quick recap of what you built. This retriever scores by counting shared words between
the query and each document. It is simple, fast, and has a serious flaw.

In [ ]:
def word_overlap_retriever(query: str, documents: list, top_k: int = 3) -> list:
    """
    Score each document by counting words it shares with the query.
    Returns top_k (score, document) tuples, sorted descending.
    """
    query_words = set(query.lower().split())
    scored = [(len(query_words & set(doc.lower().split())), doc) for doc in documents]
    scored.sort(reverse=True)
    return scored[:top_k]

---
## 2 — When it works

Keyword retrieval works well when the query and the document share the same words.

In [ ]:
knowledge_base = [
    "The Nile River flows through Egypt and is approximately 6,650 km long.",
    "Cairo is the capital city of Egypt and sits on the Nile River.",
    "The Suez Canal connects the Red Sea to the Mediterranean Sea.",
    "Embeddings convert text into high-dimensional numerical vectors.",
    "BM25 is a keyword-based ranking algorithm used in search engines.",
    "Cosine similarity measures the angle between two vectors in vector space.",
    "Large language models are trained on massive text corpora.",
    "Retrieval Augmented Generation combines search with LLM generation.",
]

# Query uses exact words from the document — works well
query = "What is the Suez Canal?"
results = word_overlap_retriever(query, knowledge_base, top_k=2)
print(f"Query: '{query}'")
for score, doc in results:
    print(f"  [overlap={score}] {doc}")

---
## 3 — Vocabulary mismatch — the core failure

The retriever finds documents by shared words.
What if the user's words and the document's words are different — but they mean the same thing?

In [ ]:
# The document says "Nile River" but the query says "river in Egypt" — partial overlap
query_partial = "Tell me about the river in Egypt."
results = word_overlap_retriever(query_partial, knowledge_base, top_k=3)
print(f"Query: '{query_partial}'")
print("Retrieved:")
for score, doc in results:
    print(f"  [overlap={score}] {doc}")

In [ ]:
# Synonym mismatch — the query uses a synonym the document doesn't contain
paraphrase_kb = [
    "The automobile industry is shifting toward electric vehicles.",
    "Sales of cars powered by gasoline declined 15% this year.",
    "Public transit usage has increased in urban areas.",
    "Bicycle commuting is growing in popularity in European cities.",
    "Aviation fuel costs rose sharply after the conflict.",
]

# 'vehicle' and 'automobile' are synonyms — the retriever doesn't know this
query_synonym = "What is happening with vehicles?"
results = word_overlap_retriever(query_synonym, paraphrase_kb, top_k=3)
print(f"Query: '{query_synonym}'")
print("Retrieved:")
for score, doc in results:
    print(f"  [overlap={score}] {doc}")

print()
print("The document about automobiles (most relevant) likely scored LOWER")
print("than documents that just happen to share common words like 'is', 'in', 'the'.")

---
## 4 — The stop-word problem

Common words like 'the', 'is', 'in', 'of' appear in almost every document.
A naive counter treats them as important signals — they are not.

In [ ]:
# Query and document share many stop words — inflates score incorrectly
noisy_kb = [
    "The cat is sitting in the corner of the room.",
    "The economy is growing in the sector of technology and in the area of finance.",
    "Suez Canal is in Egypt.",
]

query_noisy = "What is happening in the economy?"
results = word_overlap_retriever(query_noisy, noisy_kb, top_k=3)
print(f"Query: '{query_noisy}'")
for score, doc in results:
    print(f"  [overlap={score}] {doc}")

print()
print("The economy document may NOT rank first because the cat document")
print("also shares many common words like 'the', 'is', 'in'.")

---
## 5 — Failure in the RAG pipeline

A bad retriever means the wrong documents go into the augmented prompt.
A great LLM cannot fix a bad context block.

In [ ]:
def rag_with_retriever(query: str, knowledge_base: list, top_k: int = 3) -> str:
    retrieved = word_overlap_retriever(query, knowledge_base, top_k=top_k)
    context   = "\n".join([f"Document {i+1}: {doc}" for i, (_, doc) in enumerate(retrieved)])
    prompt    = f"""Answer ONLY from the documents below.

{context}

Question: {query}
Answer:"""
    return generate_with_single_input(prompt=prompt, temperature=0.0)['content']


# Ask a question where the retriever fails — wrong docs → wrong answer
large_kb = knowledge_base + paraphrase_kb + noisy_kb

q = "What is happening with cars and electric vehicles?"
print(f"Q: {q}")
print(f"\nWhat the retriever found:")
for score, doc in word_overlap_retriever(q, large_kb, top_k=3):
    print(f"  [overlap={score}] {doc}")
print(f"\nLLM answer (based on those docs):")
print(rag_with_retriever(q, large_kb))

---
## 6 — What Module 02 will fix

| Problem demonstrated | Module 02 solution |
|---------------------|-------------------|
| Synonym mismatch ("automobile" ≠ "car") | **Semantic search** — embeddings capture meaning, not just words |
| Stop-word noise ("the", "is", "in") | **BM25** — down-weights common words automatically |
| Equal weighting of rare vs common words | **TF-IDF** — rare words get higher weight |
| No way to exclude documents by attribute | **Metadata filtering** — strict yes/no conditions |
| No way to combine keyword and semantic results | **Hybrid search + RRF** — best of both |

In [ ]:
# Summary — print which queries failed and why
failures = [
    ("river in Egypt",           "document says 'Nile River' — partial match only"),
    ("What is happening with vehicles?", "document says 'automobile' — synonym not matched"),
    ("happening in the economy", "stop words inflate wrong document scores"),
]

print("Vocabulary mismatch failures in the naive retriever:")
print("=" * 60)
for query, reason in failures:
    print(f"\nQuery:  '{query}'")
    print(f"Reason: {reason}")

---
**Lab 06 complete.**

You have seen exactly where and why naive keyword retrieval fails.

Module 02 addresses every failure shown in this lab:
BM25 fixes the stop-word and word-frequency problems.
Semantic search fixes the synonym and paraphrase problems.
Hybrid search combines both.

**Assignment:** `assignments/assignment01_simple_rag.ipynb`